# Hedonic Pricing Regression
**Thesis: What Makes Vietnamese Groceries Expensive?**

This notebook runs OLS hedonic regressions to estimate implicit price premiums for product attributes (brand, import origin, health claims, pack size) in Vietnamese online grocery data.

## Pipeline
1. Load `output/product_features.csv` (built by `src/build_product_dataset.py` + `src/nlp_features.py`)
2. Filter to products with extractable size (regression requires unit-normalized DV)
3. Run pooled OLS + per-category OLS
4. Robustness: 3 date snapshot averages
5. Diagnostics: VIF, cluster-robust SE, R² decomposition

In [1]:
import sys
sys.path.insert(0, '..')  # so 'src' and 'config' are importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

## 1. Load Data

In [2]:
df = pd.read_csv('../output/product_features.csv')
print(f"Loaded: {len(df)} products across {df['parent_category'].nunique()} categories")
df.head(3)

Loaded: 2274 products across 10 categories


,product_name,unit,subcategory,parent_category,avg_final_price,avg_marked_price,days_observed,promo_rate,avg_discount_depth,price_volatility,n_price_changes,max_daily_jump_pct,ever_promoted,always_promoted,marked_price_changes,...,has_health_claim,has_freshness_claim,name_length,unit_type,pack_count,uncertain_pack_count,size_ml,size_g,ln_pack_size,price_per_100ml,price_per_100g,ln_price_per_unit,marked_price_per_100ml,marked_price_per_100g,ln_marked_price_per_unit
0,Bơ lạt Anchor gói 227g,Miếng,Bơ Sữa - Phô Mai,Sữa Tươi,124590.6250,137500.0000,64,0.5625,0.1669,0.0973,3,0.2569,1,0,1,...,0,0,5,count,1,False,NaN,227.0000,5.4250,NaN,54885.7379,10.9130,NaN,60572.6872,11.0116
1,Bơ lạt Paysan Breton hộp 250g,Hộp,Bơ Sữa - Phô Mai,Sữa Tươi,174000.0000,174000.0000,45,0.0000,NaN,0.0000,1,0.0000,0,0,1,...,0,0,6,count,1,False,NaN,250.0000,5.5215,NaN,69600.0000,11.1505,NaN,69600.0000,11.1505
2,Bơ lạt Presiden hộp 250g,Hộp,Bơ Sữa - Phô Mai,Sữa Tươi,181000.0000,181000.0000,50,0.0000,NaN,0.0000,1,0.0000,0,0,1,...,0,0,5,count,1,False,NaN,250.0000,5.5215,NaN,72400.0000,11.1900,NaN,72400.0000,11.1900


In [3]:
# Overview of key features
feature_cols = [
    'is_branded', 'is_import', 'is_house_brand',
    'has_health_claim', 'has_freshness_claim',
    'ln_pack_size', 'pack_count', 'name_length',
    'ln_price_per_unit', 'ln_marked_price_per_unit',
]
df[feature_cols].describe().round(3)

,is_branded,is_import,is_house_brand,has_health_claim,has_freshness_claim,ln_pack_size,pack_count,name_length,ln_price_per_unit,ln_marked_price_per_unit
count,2274.0000,2274.0000,2274.0000,2274.0000,2274.0000,2070.0000,2274.0000,2274.0000,2070.0000,2070.0000
mean,0.6930,0.0430,0.0090,0.0540,0.0680,5.1790,2.0390,7.7820,9.8220,9.8630
std,0.4610,0.2030,0.0960,0.2250,0.2510,1.0910,4.8980,2.5290,1.0340,1.0230
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.6930,1.0000,1.0000,7.2340,7.3060
25%,0.0000,0.0000,0.0000,0.0000,0.0000,4.5000,1.0000,6.0000,9.0600,9.1050
50%,1.0000,0.0000,0.0000,0.0000,0.0000,5.2980,1.0000,8.0000,9.7850,9.8270
75%,1.0000,0.0000,0.0000,0.0000,0.0000,5.8580,1.0000,9.0000,10.4710,10.4870
max,1.0000,1.0000,1.0000,1.0000,1.0000,8.5170,48.0000,18.0000,13.5610,13.5610


## 2. Descriptive Statistics (Table 1)
Mean prices, promotion rates, and feature rates by category.

In [4]:
table1 = df.groupby('parent_category').agg(
    n_products=('product_name', 'count'),
    avg_final_price=('avg_final_price', 'mean'),
    avg_marked_price=('avg_marked_price', 'mean'),
    promo_rate=('promo_rate', 'mean'),
    avg_discount_pct=('avg_discount_depth', 'mean'),
    pct_branded=('is_branded', 'mean'),
    pct_import=('is_import', 'mean'),
    pct_health=('has_health_claim', 'mean'),
).round(3)

table1['avg_discount_pct'] = (table1['avg_discount_pct'] * 100).round(1)
table1['promo_rate']       = (table1['promo_rate'] * 100).round(1)
table1['pct_branded']      = (table1['pct_branded'] * 100).round(1)
table1['pct_import']       = (table1['pct_import'] * 100).round(1)
table1['pct_health']       = (table1['pct_health'] * 100).round(1)

print("Table 1: Descriptive Statistics by Category")
table1

Table 1: Descriptive Statistics by Category


,n_products,avg_final_price,avg_marked_price,promo_rate,avg_discount_pct,pct_branded,pct_import,pct_health
parent_category,,,,,,,,
Bánh Kẹo,583,63353.6320,67448.5060,16.6000,19.4000,56.3000,2.9000,3.6000
Chăm Sóc Bé,58,141948.4320,147681.0340,28.8000,16.9000,93.1000,0.0000,19.0000
Gia Vị,300,67950.7790,72670.7470,35.5000,15.7000,92.7000,0.7000,3.7000
Mì - Thực Phẩm Ăn Liền,222,67002.8620,68723.9480,31.4000,14.5000,83.8000,7.2000,0.0000
Rau - Củ - Trái Cây,249,67778.3250,69269.7990,13.1000,17.8000,7.6000,13.7000,0.4000
Sữa Tươi,346,79988.2370,82121.7700,33.1000,12.5000,89.0000,3.5000,19.7000
Thực Phẩm Chế Biến,172,55189.0990,56466.8090,16.1000,16.5000,84.3000,4.7000,1.7000
Thực Phẩm Khô,182,65089.5370,67572.7050,26.9000,15.1000,75.3000,0.5000,2.2000
Thực Phẩm Đông Lạnh,137,90586.7460,92010.3910,10.1000,16.3000,73.0000,5.8000,0.0000


## 3. Regression Sample
Keep products where size was extractable (required for unit-normalized DV).

In [5]:
# Primary sample: products with extractable unit price
reg_df = df.dropna(subset=['ln_price_per_unit', 'ln_marked_price_per_unit', 'ln_pack_size', 'subcategory']).copy()
reg_df = reg_df[np.isfinite(reg_df['ln_price_per_unit'])].copy()

# Exclude extreme outliers (Winsorize at 1%/99%)
lo, hi = reg_df['ln_price_per_unit'].quantile([0.01, 0.99])
reg_df = reg_df[(reg_df['ln_price_per_unit'] >= lo) & (reg_df['ln_price_per_unit'] <= hi)]

print(f"Regression sample: {len(reg_df)} products ({len(reg_df)/len(df)*100:.1f}% of total)")
print(f"Coverage by category:")
print(reg_df.groupby('parent_category').size().rename('n_in_sample'))

Regression sample: 2028 products (89.2% of total)
Coverage by category:
parent_category
Bánh Kẹo                  550
Chăm Sóc Bé                34
Gia Vị                    294
Mì - Thực Phẩm Ăn Liền    220
Rau - Củ - Trái Cây       123
Sữa Tươi                  331
Thực Phẩm Chế Biến        167
Thực Phẩm Khô             170
Thực Phẩm Đông Lạnh       131
Trứng - Đậu Hũ              8
Name: n_in_sample, dtype: int64


## 4. Pooled OLS — Final Price DV (Table 2a)

```
ln(final_price/unit) = α + β1·is_branded + β2·is_import + β3·is_house_brand
                      + β4·ln_pack_size + β5·pack_count
                      + β6·has_health_claim + β7·has_freshness_claim
                      + β8·name_length + subcategory_FE + ε
```
SEs clustered by subcategory.

In [6]:
# 1. Define the columns used in your model
columns_in_model = [
    'ln_price_per_unit', 'is_branded', 'is_import', 'is_house_brand',
    'ln_pack_size', 'pack_count', 'has_health_claim', 'has_freshness_claim',
    'name_length', 'subcategory'
]

# 2. Drop missing values to prevent length misalignment during clustering
reg_df_clean = reg_df.dropna(subset=columns_in_model).copy()

# 3. Force the cluster group to be uniform strings (fixes the float vs str error)
reg_df_clean['subcategory'] = reg_df_clean['subcategory'].astype(str)

# 4. Run your exact same model logic on the cleaned dataframe
FORMULA = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + '
    'name_length + C(subcategory)'
)

model_final = smf.ols(FORMULA, data=reg_df_clean).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df_clean['subcategory']},
)

print(model_final.summary2())

                              Results: Ordinary least squares
Model:                       OLS                       Adj. R-squared:             0.598    
Dependent Variable:          ln_price_per_unit         AIC:                        3837.7515
Date:                        2026-03-18 15:51          BIC:                        4096.0325
No. Observations:            2028                      Log-Likelihood:             -1872.9  
Df Model:                    45                        F-statistic:                201.3    
Df Residuals:                1982                      Prob (F-statistic):         4.24e-28 
R-squared:                   0.607                     Scale:                      0.37987  
--------------------------------------------------------------------------------------------
                                             Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------------------------------
Intercep

/Users/my/Online Food Price/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 45, but rank is 8
  warnings.warn('covariance of constraints does not have full '


## 5. Pooled OLS — Marked Price DV (Table 2b)
Compare β coefficients with final_price model to see if promotions erode premiums.

In [7]:
# 1. Update the columns list to include your NEW dependent variable
columns_in_model_marked = [
    'ln_marked_price_per_unit', 'is_branded', 'is_import', 'is_house_brand',
    'ln_pack_size', 'pack_count', 'has_health_claim', 'has_freshness_claim',
    'name_length', 'subcategory'
]

# 2. Drop NaNs to ensure the model data and cluster arrays match lengths perfectly
reg_df_marked_clean = reg_df.dropna(subset=columns_in_model_marked).copy()

# 3. Force the cluster group to be uniform strings to fix the float vs. str error
reg_df_marked_clean['subcategory'] = reg_df_marked_clean['subcategory'].astype(str)

# 4. Create your new formula
FORMULA_MARKED = FORMULA.replace('ln_price_per_unit', 'ln_marked_price_per_unit')

# 5. Run the model using the CLEANED dataframe (both in data= and cov_kwds=)
model_marked = smf.ols(FORMULA_MARKED, data=reg_df_marked_clean).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df_marked_clean['subcategory']},
)

print(model_marked.summary2())

                              Results: Ordinary least squares
Model:                    OLS                            Adj. R-squared:           0.595    
Dependent Variable:       ln_marked_price_per_unit       AIC:                      3812.8478
Date:                     2026-03-18 15:51               BIC:                      4071.1288
No. Observations:         2028                           Log-Likelihood:           -1860.4  
Df Model:                 45                             F-statistic:              190.6    
Df Residuals:             1982                           Prob (F-statistic):       1.14e-27 
R-squared:                0.604                          Scale:                    0.37524  
--------------------------------------------------------------------------------------------
                                             Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------------------------------
Intercep

/Users/my/Online Food Price/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 45, but rank is 8
  warnings.warn('covariance of constraints does not have full '


## 6. Side-by-Side Coefficient Comparison (Table 2)
Marked price vs. final price premiums — shows whether promotions close the gap.

In [8]:
KEY_VARS = [
    'is_branded', 'is_import', 'is_house_brand',
    'ln_pack_size', 'pack_count',
    'has_health_claim', 'has_freshness_claim', 'name_length',
]

def extract_coefs(model, label):
    coef = model.params[KEY_VARS]
    pval = model.pvalues[KEY_VARS]
    se   = model.bse[KEY_VARS]
    stars = pval.map(lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else '')))
    result = pd.DataFrame({
        f'β ({label})': coef.round(4).astype(str) + stars,
        f'SE ({label})': se.round(4),
    })
    return result

t2 = pd.concat([
    extract_coefs(model_marked, 'marked'),
    extract_coefs(model_final,  'final'),
], axis=1)

# Add R2 row
r2_row = pd.DataFrame({
    'β (marked)': [f"{model_marked.rsquared_adj:.3f}"],
    'SE (marked)': [''],
    'β (final)':  [f"{model_final.rsquared_adj:.3f}"],
    'SE (final)':  [''],
}, index=['Adj R²'])

table2 = pd.concat([t2, r2_row])
print("Table 2: Hedonic Regression — Marked Price vs. Final Price")
print(f"N = {len(reg_df)}")
table2

Table 2: Hedonic Regression — Marked Price vs. Final Price
N = 2028


,β (marked),SE (marked),β (final),SE (final)
is_branded,0.0462,0.0898,0.0338,0.0902
is_import,0.1384,0.1269,0.1114,0.1248
is_house_brand,0.0386,0.0586,-0.1846***,0.0587
ln_pack_size,-0.3515***,0.0399,-0.363***,0.0421
pack_count,-0.0223***,0.0071,-0.0214***,0.0070
has_health_claim,-0.0311,0.1065,-0.0268,0.1152
has_freshness_claim,0.0111,0.0659,0.0044,0.0680
name_length,-0.0021,0.0135,-0.001,0.0138
Adj R²,0.595,,0.598,


## 7. Per-Category Regressions (Table 3)
Brand premium varies by category — test the Dairy vs. Veg_Fruit gradient.

In [9]:
FOCUS_CATS = ['Sữa các loại', 'Rau-Củ-Trái cây', 'Thực phẩm khô', 'Thực Phẩm Chế Biến']

# Use all categories with at least 20 products in regression sample
cat_counts = reg_df.groupby('parent_category').size()
eligible_cats = cat_counts[cat_counts >= 20].index.tolist()
print(f"Categories with ≥20 products in sample: {eligible_cats}")

SIMPLE_FORMULA = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + name_length'
)

cat_results = {}
for cat in eligible_cats:
    sub = reg_df[reg_df['parent_category'] == cat]
    if sub['subcategory'].nunique() > 1:
        formula = SIMPLE_FORMULA + ' + C(subcategory)'
    else:
        formula = SIMPLE_FORMULA
    try:
        m = smf.ols(formula, data=sub).fit()
        cat_results[cat] = m
        print(f"{cat}: n={len(sub)}, adj_R²={m.rsquared_adj:.3f}")
    except Exception as e:
        print(f"{cat}: failed — {e}")

Categories with ≥20 products in sample: ['Bánh Kẹo', 'Chăm Sóc Bé', 'Gia Vị', 'Mì - Thực Phẩm Ăn Liền', 'Rau - Củ - Trái Cây', 'Sữa Tươi', 'Thực Phẩm Chế Biến', 'Thực Phẩm Khô', 'Thực Phẩm Đông Lạnh']


Bánh Kẹo: n=550, adj_R²=0.281
Chăm Sóc Bé: n=34, adj_R²=0.762
Gia Vị: n=294, adj_R²=0.373
Mì - Thực Phẩm Ăn Liền: n=220, adj_R²=0.190
Rau - Củ - Trái Cây: n=123, adj_R²=0.554


Sữa Tươi: n=331, adj_R²=0.650


Thực Phẩm Chế Biến: n=167, adj_R²=0.309
Thực Phẩm Khô: n=170, adj_R²=0.805
Thực Phẩm Đông Lạnh: n=131, adj_R²=0.504


In [10]:
# Compile brand + import premiums across categories
rows = []
for cat, m in cat_results.items():
    row = {'category': cat, 'n': int(m.nobs), 'adj_R2': round(m.rsquared_adj, 3)}
    for var in ['is_branded', 'is_import', 'is_house_brand', 'has_health_claim']:
        if var in m.params:
            row[var]          = round(m.params[var], 4)
            row[f'{var}_pval'] = round(m.pvalues[var], 4)
    rows.append(row)

table3 = pd.DataFrame(rows).set_index('category')
print("Table 3: Cross-Category Premium Estimates (ln_price_per_unit DV)")
table3

Table 3: Cross-Category Premium Estimates (ln_price_per_unit DV)


,n,adj_R2,is_branded,is_branded_pval,is_import,is_import_pval,is_house_brand,is_house_brand_pval,has_health_claim,has_health_claim_pval
category,,,,,,,,,,
Bánh Kẹo,550,0.2810,0.1131,0.0433,0.0903,0.6146,0.0000,0.0000,0.1668,0.2823
Chăm Sóc Bé,34,0.7620,0.4188,0.2890,0.0000,0.0000,-0.0000,0.0053,-0.0366,0.8660
Gia Vị,294,0.3730,-0.2986,0.0456,-0.1849,0.6829,0.0000,0.0000,0.3019,0.1343
Mì - Thực Phẩm Ăn Liền,220,0.1900,0.0143,0.8925,0.0758,0.6066,0.0000,0.0000,0.0000,NaN
Rau - Củ - Trái Cây,123,0.5540,-0.1120,0.5769,1.1953,0.0000,-0.3934,0.0409,-0.0000,0.0335
Sữa Tươi,331,0.6500,-0.0586,0.5959,-0.0198,0.9172,-0.0000,0.1236,-0.2004,0.0384
Thực Phẩm Chế Biến,167,0.3090,0.1671,0.2320,-0.6241,0.0176,0.0000,0.7619,0.0137,0.9690
Thực Phẩm Khô,170,0.8050,-0.0969,0.3663,0.8791,0.1135,-0.0000,0.0217,0.2689,0.3686
Thực Phẩm Đông Lạnh,131,0.5040,-0.0864,0.2726,-0.1450,0.3164,-0.0000,0.0000,0.0000,NaN


## 8. Diagnostics

In [11]:
# VIF — multicollinearity check
X_vif = reg_df[[
    'is_branded', 'is_import', 'is_house_brand',
    'ln_pack_size', 'pack_count',
    'has_health_claim', 'has_freshness_claim', 'name_length',
]].dropna()

X_vif_const = sm.add_constant(X_vif)
vif_df = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif_const.values, i+1) for i in range(X_vif.shape[1])],
})
print("VIF (rule of thumb: VIF > 10 indicates multicollinearity)")
print(vif_df.to_string(index=False))

VIF (rule of thumb: VIF > 10 indicates multicollinearity)
            feature    VIF
         is_branded 1.1001
          is_import 1.0110
     is_house_brand 1.0236
       ln_pack_size 1.0405
         pack_count 1.1732
   has_health_claim 1.0596
has_freshness_claim 1.0876
        name_length 1.3465


In [12]:
# Partial F-tests: brand block and health claim block
RESTRICTED_NO_BRAND = (
    'ln_price_per_unit ~ '
    'is_import + ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + name_length + C(subcategory)'
)
RESTRICTED_NO_HEALTH = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + name_length + C(subcategory)'
)

m_no_brand  = smf.ols(RESTRICTED_NO_BRAND,  data=reg_df).fit()
m_no_health = smf.ols(RESTRICTED_NO_HEALTH, data=reg_df).fit()
m_full      = smf.ols(FORMULA, data=reg_df).fit()  # no cluster for F-test

# F-statistic: (ΔR²/Δk) / ((1-R²_full)/(n-k_full))
def f_test(r_full, r_restricted, k_added, n, k_full):
    return ((r_full - r_restricted) / k_added) / ((1 - r_full) / (n - k_full - 1))

n = len(reg_df)
k = m_full.df_model

f_brand  = f_test(m_full.rsquared, m_no_brand.rsquared,  2, n, k)  # is_branded + is_house_brand
f_health = f_test(m_full.rsquared, m_no_health.rsquared, 2, n, k)  # has_health + has_fresh

print(f"Partial F-test — Brand block (is_branded + is_house_brand): F={f_brand:.2f}")
print(f"Partial F-test — Health claim block: F={f_health:.2f}")
print(f"(Critical value at α=0.05, df_num=2: ~3.0)")

Partial F-test — Brand block (is_branded + is_house_brand): F=0.87
Partial F-test — Health claim block: F=0.08
(Critical value at α=0.05, df_num=2: ~3.0)


## 9. Robustness: 3 Date Snapshots
Re-run pooled regression on Dec 2025, Jan 2026, and Feb 2026 monthly averages separately.
Check if key coefficients are stable (β₁ brand premium should vary < 20%).

In [13]:
import sys, importlib
sys.path.insert(0, '..')

from pathlib import Path
from src.utils import PROJECT_ROOT, extract_date
from src.nlp_features import extract_features

CATEGORIES = [
    "Dairy", "Baby_product", "Veg_Fruit", "Confectionary",
    "Dry_Food", "Egg_and_soy", "Frozen", "Instant_food",
    "Processed_food", "Spice",
]
NEEDED_COLS = {'product_name', 'unit', 'final_price', 'marked_price'}
MONTH_RANGES = {
    'Dec-2025': ('2025-12-01', '2025-12-31'),
    'Jan-2026': ('2026-01-01', '2026-01-31'),
    'Feb-2026': ('2026-02-01', '2026-02-28'),
}

def load_snapshot(month_start, month_end):
    """Build product-level dataset for a specific date window."""
    all_frames = []
    for cat in CATEGORIES:
        folder = PROJECT_ROOT / 'data' / cat
        csv_folder = folder / 'CSV'
        lookup_file = folder / 'cat_lookup_table.csv'
        frames = []
        for f in sorted(csv_folder.glob('*.csv')):
            d = extract_date(f.name)
            if not (month_start <= d <= month_end):
                continue
            try:
                df_ = pd.read_csv(f, usecols=lambda c: c in NEEDED_COLS)
                frames.append(df_)
            except Exception:
                pass
        if not frames:
            continue
        daily = pd.concat(frames, ignore_index=True)
        daily['final_price'] = pd.to_numeric(daily['final_price'], errors='coerce')
        daily['marked_price'] = pd.to_numeric(daily['marked_price'], errors='coerce')
        daily = daily[daily['final_price'] > 0]
        if lookup_file.exists():
            lkp = pd.read_csv(lookup_file, usecols=['product_name', 'subcategory', 'parent_category'])
            lkp = lkp.drop_duplicates(subset=['product_name'], keep='last')
            daily = daily.merge(lkp, on='product_name', how='left')
        else:
            daily['subcategory'] = 'Gia Vị'
            daily['parent_category'] = 'Gia Vị'
        grp = daily.groupby(['product_name', 'unit', 'subcategory', 'parent_category'], dropna=False)
        agg = grp.agg(
            avg_final_price=('final_price', 'mean'),
            avg_marked_price=('marked_price', 'mean'),
        ).reset_index()
        all_frames.append(agg)
    if not all_frames:
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True)


snapshot_models = {}
for month, (start, end) in MONTH_RANGES.items():
    snap = load_snapshot(start, end)
    if snap.empty:
        print(f"{month}: no data")
        continue
    snap_feat = extract_features(snap)
    snap_reg = snap_feat.dropna(subset=['ln_price_per_unit', 'ln_pack_size', 'subcategory'])
    snap_reg = snap_reg[np.isfinite(snap_reg['ln_price_per_unit'])]
    if len(snap_reg) < 30:
        print(f"{month}: too few obs ({len(snap_reg)})")
        continue
    try:
        m = smf.ols(FORMULA, data=snap_reg).fit(
            cov_type='cluster',
            cov_kwds={'groups': snap_reg['subcategory'].astype(str)},
        )
        snapshot_models[month] = m
        print(f"{month}: n={len(snap_reg)}, adj_R²={m.rsquared_adj:.3f}, β_brand={m.params.get('is_branded', float('nan')):.4f}")
    except Exception as e:
        print(f"{month}: {e}")

# Stability check: β_brand across months
brand_betas = {m: mod.params.get('is_branded', np.nan) for m, mod in snapshot_models.items()}
print("\nBrand premium stability across months:")
print(pd.Series(brand_betas))

Dec-2025: n=1240, adj_R²=0.624, β_brand=-0.0972


Jan-2026: n=1250, adj_R²=0.626, β_brand=-0.0954


Feb-2026: n=1971, adj_R²=0.613, β_brand=0.0152

Brand premium stability across months:
Dec-2025   -0.0972
Jan-2026   -0.0954
Feb-2026    0.0152
dtype: float64


## 10. Export Tables for Thesis

In [14]:
out = PROJECT_ROOT / 'output'

table1.to_csv(out / 'thesis_table1_descriptive.csv', encoding='utf-8-sig')
table2.to_csv(out / 'thesis_table2_pooled_regression.csv', encoding='utf-8-sig')
table3.to_csv(out / 'thesis_table3_category_regression.csv', encoding='utf-8-sig')
vif_df.to_csv(out / 'thesis_table_vif.csv', index=False, encoding='utf-8-sig')

print("Saved 4 tables to output/")

Saved 4 tables to output/
